# 04. Paoay map

The frozen model is applied to every Paoay study pixel for the most complete Cloud Score+ year. Low-confidence pixels become Unknown, and cropland is flagged from the land-activity map. The map is exploratory: no local labels exist.

**Input:** current models. **Output:** `current_paoay.json`, `crop_map.tif`, `probabilities.tif`.

## Setup

In [ ]:
from pathlib import Path
import json
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").is_file():
    if root == root.parent:
        raise RuntimeError("Open this notebook inside the project repository")
    root = root.parent
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # experiments package

import pandas as pd
from IPython.display import display
from experiments.improved_v1 import workflow
pd.set_option("display.width", 200)

## Map

In [ ]:
models_handoff = ""  # Optional models receipt; blank uses the current pointer
receipt = workflow.paoay_map(root, models_handoff or None)
summary = json.loads((receipt.parent / "paoay_summary.json").read_text())
print("Paoay receipt:", receipt)
display(pd.Series({k: v for k, v in summary.items() if not isinstance(v, dict)}).to_frame("value"))
display(pd.DataFrame({"map (confident cropland)": summary["crop_shares_of_confident_cropland"],
                      "Benguet 2024": summary["benguet_2024_shares"]}).round(3))
display(pd.read_csv(receipt.parent / "area_summary.csv"))

## Preview

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
with rasterio.open(receipt.parent / "crop_map.tif") as raster:
    codes, cropland = raster.read(1), raster.read(2)
colors = ["#fcfcfb", "#eb6834", "#1baf7a", "#eda100", "#c3c2b7", "#e4e3df"]
figure, axes = plt.subplots(1, 2, figsize=(13, 6))
axes[0].imshow(codes, cmap=ListedColormap(colors), vmin=0, vmax=5, interpolation="nearest")
axes[0].set_title("Crop map (1 carrot, 2 cabbage, 3 potato, 4 Unknown, 5 no features)", fontsize=9)
axes[1].imshow(cropland, cmap=ListedColormap(["#fcfcfb", "#e4e3df", "#1baf7a"]), vmin=0, vmax=2, interpolation="nearest")
axes[1].set_title("Land-activity cropland (green)", fontsize=9)
for axis in axes:
    axis.axis("off")
figure.tight_layout()
figure.savefig(receipt.parent / "preview.png", dpi=150)
display(figure)
plt.close(figure)